# Customer Sales & Churn Analytics

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026**

This notebook packages the complete project source code in Jupyter Notebook (`.ipynb`) format for submission.

## Project components

- `data_cleaning.py` — data loading, validation, and quality reporting
- `analytics.py` — descriptive sales analytics and filtering
- `churn_model.py` — RFM feature engineering + Logistic Regression churn model
- `app.py` — Streamlit dashboard
- `requirements.txt` — Python dependencies
- `README.md` — project documentation

> **Data note:** `Clean_Data.csv` and `churn_risk_table.csv` are project data artifacts and are expected to be present in the project folder when the notebook/project is executed.

## 1. Environment check

The project documentation specifies a Python 3.11 environment and the pinned dependencies in `requirements.txt`.

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

for name in ["numpy", "pandas", "scipy", "sklearn", "plotly", "streamlit"]:
    try:
        mod = __import__(name)
        print(f"{name}:", getattr(mod, "__version__", "installed"))
    except Exception as e:
        print(f"{name}: IMPORT ERROR ->", e)

## 2. Complete source — `data_cleaning.py`

In [ ]:
%%writefile data_cleaning.py
"""
data_cleaning.py
================
Loads Clean_Data.csv, validates every column, documents all data-quality
issues, and returns a cleaned DataFrame together with a structured quality
report.

Design rules applied here:
- The original source file is never modified.
- Duplicate rows are DETECTED and REPORTED but NOT removed, so that the
  churn-model methodology stays consistent with the Masterclass 3 baseline.
- Invalid dates are flagged (Order_Date_Valid = False) but the rows are kept.
- Missing Revenue values remain NaN; they are never replaced with zero or
  an arbitrary average.
- Negative Profit values (losses) are preserved as-is.
- "Unknown Customer", "Unknown Product", and "Unknown Region" strings are
  flagged but the rows are retained for sales analytics.
"""

import pandas as pd
import numpy as np


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

CSV_PATH = "Clean_Data.csv"

VALID_CATEGORIES = {"Apparel", "Beauty", "Electronics", "Furniture", "Home"}
VALID_REGIONS = {
    "Bangalore", "Chennai", "Delhi", "Hyderabad",
    "Kolkata", "Mumbai", "Pune",
}
VALID_CUSTOMER_TYPES = {"New Customer", "Repeat Customer", "Unknown Customer"}
VALID_PROFIT_STATUSES = {"Profit", "Loss", "Break-even"}

MONTH_ORDER = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _flag_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add an `Is_Duplicate` boolean column.
    The FIRST occurrence of every duplicated row is marked False;
    all subsequent occurrences are marked True.
    Rows are never dropped — detection only.
    """
    df["Is_Duplicate"] = df.duplicated(keep="first")
    return df


def _parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Attempt to parse Order_Date with multiple common formats.
    Rows where parsing fails (e.g. the literal string "Invalid Date") get
    NaT in Order_Date and Order_Date_Valid = False.
    The original raw string is preserved in Order_Date_Raw.
    """
    df["Order_Date_Raw"] = df["Order_Date"].astype(str).str.strip()

    parsed = pd.to_datetime(df["Order_Date_Raw"], format="%m/%d/%Y", errors="coerce")
    # Fallback: try dayfirst for any still-NaT rows
    mask_nat = parsed.isna()
    if mask_nat.any():
        fallback = pd.to_datetime(
            df.loc[mask_nat, "Order_Date_Raw"], dayfirst=True, errors="coerce"
        )
        parsed[mask_nat] = fallback

    df["Order_Date"] = parsed
    df["Order_Date_Valid"] = df["Order_Date"].notna()
    return df


def _clean_revenue_profit(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert Revenue and Profit to numeric.
    - Blank Revenue → NaN  (never replaced with zero or a mean).
    - Zero Revenue rows are flagged with Revenue_Is_Zero.
    - Negative Profit values are preserved; they represent real losses.
    """
    df["Revenue"] = pd.to_numeric(df["Revenue"], errors="coerce")
    df["Profit"] = pd.to_numeric(df["Profit"], errors="coerce")

    df["Revenue_Missing"] = df["Revenue"].isna()
    df["Revenue_Is_Zero"] = df["Revenue"].eq(0)
    return df


def _flag_unknown_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add boolean flag columns for anomalous string values that are kept in
    the dataset but need to be excludable for specific analyses.
    """
    df["Customer_Unknown"] = (
        df["Customer_ID"].astype(str).str.strip() == "Unknown Customer"
    )
    df["Product_Unknown"] = (
        df["Product"].astype(str).str.strip() == "Unknown Product"
    )
    df["Region_Unknown"] = (
        df["Region"].astype(str).str.strip() == "Unknown Region"
    )
    return df


def _standardise_text(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strip leading/trailing whitespace from all object columns.
    Standardise Customer_Type casing to title-case.
    """
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()

    df["Customer_Type"] = df["Customer_Type"].str.title()
    df["Category"] = df["Category"].str.title()
    df["Region"] = df["Region"].str.title()
    df["Profit_Status"] = df["Profit_Status"].str.strip()
    return df


def _validate_profit_status(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cross-check the Profit_Status label against the actual Profit value.
    Adds a Profit_Status_Mismatch column for rows where the label disagrees
    with the sign of Profit.
    """
    def expected_status(profit):
        if pd.isna(profit):
            return None
        if profit > 0:
            return "Profit"
        if profit < 0:
            return "Loss"
        return "Break-even"

    expected = df["Profit"].apply(expected_status)
    df["Profit_Status_Mismatch"] = (
        expected.notna() & (df["Profit_Status"] != expected)
    )
    return df


def _add_month_order(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add a numeric Month_Num column for calendar ordering.
    Rows with Month = 'Unknown' receive NaN.
    """
    month_map = {m: i + 1 for i, m in enumerate(MONTH_ORDER)}
    df["Month_Num"] = df["Month"].map(month_map)
    return df


# ---------------------------------------------------------------------------
# Quality report builder
# ---------------------------------------------------------------------------

def _build_quality_report(df: pd.DataFrame) -> dict:
    """
    Compile a structured dictionary of all data-quality issues found.
    This is displayed in the Streamlit dashboard's Data Quality expander.
    """
    total = len(df)

    report = {
        "total_rows": total,
        "total_columns": len(df.columns),

        # Date issues
        "invalid_date_count": int((~df["Order_Date_Valid"]).sum()),
        "invalid_date_pct": round(100 * (~df["Order_Date_Valid"]).mean(), 1),

        # Duplicate rows (detected, NOT removed)
        "duplicate_row_count": int(df["Is_Duplicate"].sum()),
        "duplicate_row_note": (
            "Duplicates detected and reported. NOT removed — required for "
            "methodological consistency with Masterclass 3 baseline results."
        ),

        # Revenue issues
        "missing_revenue_count": int(df["Revenue_Missing"].sum()),
        "missing_revenue_pct": round(100 * df["Revenue_Missing"].mean(), 1),
        "zero_revenue_count": int(df["Revenue_Is_Zero"].sum()),
        "zero_revenue_note": (
            "Missing Revenue values remain NaN. They are excluded from "
            "revenue calculations but the rows are preserved in the dataset."
        ),

        # Unknown / anomalous strings
        "unknown_customer_count": int(df["Customer_Unknown"].sum()),
        "unknown_product_count": int(df["Product_Unknown"].sum()),
        "unknown_region_count": int(df["Region_Unknown"].sum()),

        # Profit breakdown
        "loss_rows": int((df["Profit_Status"] == "Loss").sum()),
        "break_even_rows": int((df["Profit_Status"] == "Break-even").sum()),
        "profit_status_mismatch_count": int(df["Profit_Status_Mismatch"].sum()),

        # Month quality
        "unknown_month_count": int((df["Month"] == "Unknown").sum()),

        # Valid-date subset size (used by churn model)
        "valid_date_rows": int(df["Order_Date_Valid"].sum()),
        "valid_date_and_revenue_rows": int(
            (df["Order_Date_Valid"] & ~df["Revenue_Missing"]).sum()
        ),
    }
    return report


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def load_and_clean(filepath: str = CSV_PATH) -> tuple[pd.DataFrame, dict]:
    """
    Load Clean_Data.csv, apply all validation and cleaning steps, and return:

        df_clean      : cleaned DataFrame (all original rows retained)
        quality_report: structured dict of data-quality findings

    Parameters
    ----------
    filepath : str
        Path to the CSV file. Defaults to CSV_PATH.

    Returns
    -------
    tuple[pd.DataFrame, dict]
    """
    # --- Load ---
    df = pd.read_csv(
        filepath,
        dtype=str,           # read everything as string first; convert below
        keep_default_na=False,  # prevent pandas treating "NA"/"" as NaN yet
        skipinitialspace=True,
    )

    # Replace genuine empty strings with NaN now that we control the types
    df.replace("", np.nan, inplace=True)

    # --- Pipeline ---
    df = _standardise_text(df)
    df = _flag_duplicates(df)
    df = _parse_dates(df)
    df = _clean_revenue_profit(df)
    df = _flag_unknown_fields(df)
    df = _validate_profit_status(df)
    df = _add_month_order(df)

    # Ensure Quantity is integer-safe numeric
    df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

    # Build the quality report before returning
    quality_report = _build_quality_report(df)

    return df, quality_report


# ---------------------------------------------------------------------------
# Convenience: sub-filtered views used by other modules
# ---------------------------------------------------------------------------

def get_valid_date_subset(df: pd.DataFrame) -> pd.DataFrame:
    """Return rows with a successfully parsed Order_Date."""
    return df[df["Order_Date_Valid"]].copy()


def get_churn_eligible_subset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return rows suitable for churn feature engineering:
    - Valid Order_Date
    - Non-missing Revenue
    - Known Customer_ID (excludes 'Unknown Customer')
    """
    return df[
        df["Order_Date_Valid"]
        & ~df["Revenue_Missing"]
        & ~df["Customer_Unknown"]
    ].copy()


# ---------------------------------------------------------------------------
# Quick self-test (run directly: python data_cleaning.py)
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    df_clean, report = load_and_clean()

    print("=" * 60)
    print("DATA CLEANING — SELF-TEST")
    print("=" * 60)
    print(f"  Rows loaded            : {report['total_rows']}")
    print(f"  Columns                : {report['total_columns']}")
    print()
    print("  --- Date Quality ---")
    print(f"  Invalid dates          : {report['invalid_date_count']} ({report['invalid_date_pct']}%)")
    print(f"  Valid-date rows        : {report['valid_date_rows']}")
    print()
    print("  --- Revenue Quality ---")
    print(f"  Missing Revenue        : {report['missing_revenue_count']} ({report['missing_revenue_pct']}%)")
    print(f"  Zero Revenue           : {report['zero_revenue_count']}")
    print()
    print("  --- Duplicates ---")
    print(f"  Duplicate rows detected: {report['duplicate_row_count']} (NOT removed)")
    print()
    print("  --- Unknown / Anomalous Strings ---")
    print(f"  Unknown Customer       : {report['unknown_customer_count']}")
    print(f"  Unknown Product        : {report['unknown_product_count']}")
    print(f"  Unknown Region         : {report['unknown_region_count']}")
    print()
    print("  --- Profit Quality ---")
    print(f"  Loss rows              : {report['loss_rows']}")
    print(f"  Break-even rows        : {report['break_even_rows']}")
    print(f"  Profit_Status mismatch : {report['profit_status_mismatch_count']}")
    print()
    print("  --- Churn-model eligible rows ---")
    print(f"  Valid date + Revenue   : {report['valid_date_and_revenue_rows']}")
    print()
    print("  dtypes of key columns:")
    for col in ["Order_Date", "Revenue", "Profit", "Quantity"]:
        print(f"    {col:20s}: {df_clean[col].dtype}")
    print()
    print("  Flag columns added:", [c for c in df_clean.columns if c not in [
        "Order_ID","Order_Date","Customer_ID","Product","Category",
        "Region","Quantity","Revenue","Profit","Customer_Type","Month","Profit_Status"
    ]])
    print("=" * 60)
    print("data_cleaning.py: PASS")


## 2. Complete source — `analytics.py`

In [ ]:
%%writefile analytics.py
"""
analytics.py
============
Descriptive sales analytics for the Customer Sales & Churn Analytics project.

All functions accept the cleaned DataFrame produced by data_cleaning.load_and_clean()
and return plain DataFrames ready for Plotly charts or KPI display in Streamlit.

Design rules:
- Revenue calculations exclude rows where Revenue is NaN (missing values are
  never filled with zero or a mean).
- Zero-Revenue rows are included in counts but their ₹0 contribution is
  honest — they are NOT excluded from aggregations.
- "Unknown Region" rows are excluded from region aggregations so charts
  show only the seven known cities.
- "Unknown Product" rows are excluded from product aggregations.
- Duplicate rows are NOT removed here; data_cleaning.py detected them but
  the Masterclass 3 methodology keeps them in the pipeline.
- Month ordering follows the calendar (January → December), not alphabetical.
- All monetary values are in Indian Rupees (₹).
"""

import pandas as pd
import numpy as np

# Calendar month order for sorting
MONTH_ORDER = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]


# ---------------------------------------------------------------------------
# KPI summary
# ---------------------------------------------------------------------------

def get_kpis(df: pd.DataFrame) -> dict:
    """
    Return the five top-level KPIs as a plain dict.

    - Total Revenue : sum of non-NaN Revenue values
    - Total Profit  : sum of all Profit values (negatives preserved → losses)
    - Total Orders  : count of rows (each row = one order)
    - Total Quantity: sum of Quantity
    - Unique Customers: count of distinct Customer_IDs excluding 'Unknown Customer'

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned DataFrame from data_cleaning.load_and_clean().

    Returns
    -------
    dict with keys: total_revenue, total_profit, total_orders,
                    total_quantity, unique_customers
    """
    revenue_rows = df[~df["Revenue_Missing"]]

    total_revenue = revenue_rows["Revenue"].sum()
    total_profit = df["Profit"].sum()
    total_orders = len(df)
    total_quantity = df["Quantity"].sum()
    unique_customers = df.loc[~df["Customer_Unknown"], "Customer_ID"].nunique()

    return {
        "total_revenue": total_revenue,
        "total_profit": total_profit,
        "total_orders": total_orders,
        "total_quantity": int(total_quantity),
        "unique_customers": unique_customers,
    }


# ---------------------------------------------------------------------------
# Monthly revenue trend
# ---------------------------------------------------------------------------

def revenue_by_month(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate Revenue by calendar month using only rows that have:
      - a valid Order_Date (Order_Date_Valid == True)
      - a non-NaN Revenue value

    The returned DataFrame is sorted in calendar order and includes
    a Month_Num column so Plotly can keep the correct x-axis order.

    Returns
    -------
    pd.DataFrame with columns: Month, Month_Num, Revenue
    """
    eligible = df[df["Order_Date_Valid"] & ~df["Revenue_Missing"]].copy()

    # Extract month name from the parsed date (more reliable than the Month column
    # which contains 'Unknown' for the 1 194 invalid-date rows)
    eligible["Month_Name"] = eligible["Order_Date"].dt.strftime("%B")

    monthly = (
        eligible
        .groupby("Month_Name", as_index=False)["Revenue"]
        .sum()
        .rename(columns={"Month_Name": "Month"})
    )

    # Add sort key and enforce calendar order
    month_map = {m: i + 1 for i, m in enumerate(MONTH_ORDER)}
    monthly["Month_Num"] = monthly["Month"].map(month_map)
    monthly = monthly.sort_values("Month_Num").reset_index(drop=True)

    return monthly[["Month", "Month_Num", "Revenue"]]


# ---------------------------------------------------------------------------
# Revenue by Category
# ---------------------------------------------------------------------------

def revenue_by_category(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate total Revenue by Category, excluding NaN Revenue rows.

    Returns
    -------
    pd.DataFrame with columns: Category, Revenue
        sorted descending by Revenue.
    """
    eligible = df[~df["Revenue_Missing"]].copy()

    cat = (
        eligible
        .groupby("Category", as_index=False)["Revenue"]
        .sum()
        .sort_values("Revenue", ascending=False)
        .reset_index(drop=True)
    )
    return cat


# ---------------------------------------------------------------------------
# Revenue by Region
# ---------------------------------------------------------------------------

def revenue_by_region(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate total Revenue by Region.

    'Unknown Region' rows are excluded so charts show only the seven
    known cities. Missing Revenue values are also excluded.

    Returns
    -------
    pd.DataFrame with columns: Region, Revenue
        sorted descending by Revenue.
    """
    eligible = df[~df["Revenue_Missing"] & ~df["Region_Unknown"]].copy()

    reg = (
        eligible
        .groupby("Region", as_index=False)["Revenue"]
        .sum()
        .sort_values("Revenue", ascending=False)
        .reset_index(drop=True)
    )
    return reg


# ---------------------------------------------------------------------------
# Revenue by Product
# ---------------------------------------------------------------------------

def revenue_by_product(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """
    Aggregate total Revenue by Product, returning the top N products.

    'Unknown Product' rows are excluded. Missing Revenue values are excluded.

    Parameters
    ----------
    top_n : int
        Number of top products to return (default 10).

    Returns
    -------
    pd.DataFrame with columns: Product, Revenue
        sorted descending by Revenue (highest first).
    """
    eligible = df[~df["Revenue_Missing"] & ~df["Product_Unknown"]].copy()

    prod = (
        eligible
        .groupby("Product", as_index=False)["Revenue"]
        .sum()
        .sort_values("Revenue", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return prod


# ---------------------------------------------------------------------------
# Profit by Product
# ---------------------------------------------------------------------------

def profit_by_product(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """
    Aggregate total Profit by Product, returning the top N products
    by absolute profit contribution.

    Negative Profit values (losses) are preserved. 'Unknown Product'
    rows are excluded.

    Parameters
    ----------
    top_n : int
        Number of top products to return (default 10).

    Returns
    -------
    pd.DataFrame with columns: Product, Profit
        sorted descending by Profit.
    """
    eligible = df[~df["Product_Unknown"]].copy()

    prod = (
        eligible
        .groupby("Product", as_index=False)["Profit"]
        .sum()
        .sort_values("Profit", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return prod


# ---------------------------------------------------------------------------
# Customer Type distribution
# ---------------------------------------------------------------------------

def customer_type_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """
    Count of orders by Customer_Type.

    All three types are included: New Customer, Repeat Customer,
    Unknown Customer (so the chart honestly reflects data quality).

    Returns
    -------
    pd.DataFrame with columns: Customer_Type, Count
        sorted descending by Count.
    """
    dist = (
        df
        .groupby("Customer_Type", as_index=False)
        .size()
        .rename(columns={"size": "Count"})
        .sort_values("Count", ascending=False)
        .reset_index(drop=True)
    )
    return dist


# ---------------------------------------------------------------------------
# Profit / Loss / Break-even distribution
# ---------------------------------------------------------------------------

def profit_loss_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """
    Count of orders by Profit_Status (Profit / Loss / Break-even).

    Returns
    -------
    pd.DataFrame with columns: Profit_Status, Count
    """
    dist = (
        df
        .groupby("Profit_Status", as_index=False)
        .size()
        .rename(columns={"size": "Count"})
        .sort_values("Count", ascending=False)
        .reset_index(drop=True)
    )
    return dist


# ---------------------------------------------------------------------------
# Filtered DataFrame helper (used by Streamlit sidebar filters)
# ---------------------------------------------------------------------------

def apply_filters(
    df: pd.DataFrame,
    regions: list | None = None,
    categories: list | None = None,
    customer_types: list | None = None,
) -> pd.DataFrame:
    """
    Apply sidebar filter selections to df and return a filtered copy.

    None or empty list means "no filter applied" (all values kept).

    Parameters
    ----------
    regions        : list of region strings to keep
    categories     : list of category strings to keep
    customer_types : list of customer type strings to keep

    Returns
    -------
    pd.DataFrame (filtered copy)
    """
    filtered = df.copy()

    if regions:
        filtered = filtered[filtered["Region"].isin(regions)]
    if categories:
        filtered = filtered[filtered["Category"].isin(categories)]
    if customer_types:
        filtered = filtered[filtered["Customer_Type"].isin(customer_types)]

    return filtered


# ---------------------------------------------------------------------------
# Quick self-test (run directly: python analytics.py)
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    from data_cleaning import load_and_clean

    df_clean, quality_report = load_and_clean()

    print("=" * 60)
    print("ANALYTICS - SELF-TEST")
    print("=" * 60)

    # --- KPIs ---
    kpis = get_kpis(df_clean)
    print("\n[KPIs]")
    print(f"  Total Revenue    : Rs.{kpis['total_revenue']:,.0f}")
    print(f"  Total Profit     : Rs.{kpis['total_profit']:,.0f}")
    print(f"  Total Orders     : {kpis['total_orders']:,}")
    print(f"  Total Quantity   : {kpis['total_quantity']:,}")
    print(f"  Unique Customers : {kpis['unique_customers']:,}")

    # --- Monthly Revenue ---
    monthly = revenue_by_month(df_clean)
    print(f"\n[Monthly Revenue - {len(monthly)} months with valid-date data]")
    print(monthly.to_string(index=False))

    # --- Revenue by Category ---
    cat = revenue_by_category(df_clean)
    print(f"\n[Revenue by Category - {len(cat)} categories]")
    print(cat.to_string(index=False))

    # --- Revenue by Region ---
    reg = revenue_by_region(df_clean)
    print(f"\n[Revenue by Region - {len(reg)} regions]")
    print(reg.to_string(index=False))

    # --- Top 10 Products by Revenue ---
    rev_prod = revenue_by_product(df_clean, top_n=10)
    print(f"\n[Top 10 Products by Revenue]")
    print(rev_prod.to_string(index=False))

    # --- Top 10 Products by Profit ---
    prof_prod = profit_by_product(df_clean, top_n=10)
    print(f"\n[Top 10 Products by Profit]")
    print(prof_prod.to_string(index=False))

    # --- Customer Type Distribution ---
    ct = customer_type_distribution(df_clean)
    print(f"\n[Customer Type Distribution]")
    print(ct.to_string(index=False))

    # --- Profit/Loss Distribution ---
    pl = profit_loss_distribution(df_clean)
    print(f"\n[Profit/Loss/Break-even Distribution]")
    print(pl.to_string(index=False))

    # --- Filter test ---
    filtered = apply_filters(df_clean, regions=["Delhi", "Mumbai"], categories=["Electronics"])
    print(f"\n[Filter test - Delhi+Mumbai, Electronics]: {len(filtered)} rows")

    print("\n" + "=" * 60)
    print("analytics.py: PASS")


## 2. Complete source — `churn_model.py`

In [ ]:
%%writefile churn_model.py
"""
churn_model.py
==============
Leakage-safe customer churn model for the Customer Sales & Churn Analytics project.

Methodology (consistent with Masterclass 3):
---------------------------------------------
1. Use only transactions with BOTH a valid Order_Date AND non-missing Revenue.
   Exclude 'Unknown Customer' rows — they cannot represent an identifiable customer.

2. Determine the observation window:
       latest_date        = maximum valid Order_Date in the eligible set
       historical_cutoff  = latest_date - 180 days
   Historical window : Order_Date <= historical_cutoff   (features built here)
   Future window     : Order_Date >  historical_cutoff   (target defined here)

3. Build customer-level RFM features from the HISTORICAL window only:
       Recency           = days from historical_cutoff to customer's last purchase
       Frequency         = number of orders in the historical window
       Monetary_Value    = total Revenue in the historical window
       Average_Order_Value = Monetary_Value / Frequency

4. Define Churn_Status from the FUTURE window only:
       Customer has a purchase in future window  ->  Churn_Status = 0 (active)
       Customer has NO purchase in future window ->  Churn_Status = 1 (churned)

5. Train / evaluate model:
       StandardScaler  ->  LogisticRegression(max_iter=1000)
       train_test_split(test_size=0.25, random_state=42, stratify=y)
       Metrics: Accuracy, Precision, Recall, Confusion Matrix

6. Score ALL customers in the historical window (not just the test split):
       Churn_Probability -> Risk_Level -> Recommended_Action
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# ---------------------------------------------------------------------------
# Risk-level thresholds
# ---------------------------------------------------------------------------
RISK_LOW_MAX    = 0.40   # probability < 0.40  -> Low Risk
RISK_MEDIUM_MAX = 0.70   # 0.40 <= prob < 0.70 -> Medium Risk
                          # prob >= 0.70         -> High Risk

RECOMMENDED_ACTIONS = {
    "Low Risk":    "Monitor regularly",
    "Medium Risk": "Send re-engagement campaign",
    "High Risk":   "Immediate retention offer required",
}

FEATURES = ["Recency", "Frequency", "Monetary_Value", "Average_Order_Value"]


# ---------------------------------------------------------------------------
# Step 1 – Filter eligible rows
# ---------------------------------------------------------------------------

def _get_eligible(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return rows that are usable for churn modelling:
      - valid Order_Date
      - non-missing Revenue
      - known Customer_ID (not 'Unknown Customer')
    """
    return df[
        df["Order_Date_Valid"]
        & ~df["Revenue_Missing"]
        & ~df["Customer_Unknown"]
    ].copy()


# ---------------------------------------------------------------------------
# Step 2 – Determine cutoff dates
# ---------------------------------------------------------------------------

def _get_cutoffs(eligible: pd.DataFrame) -> tuple[pd.Timestamp, pd.Timestamp]:
    """
    Returns (historical_cutoff, latest_date).
    historical_cutoff = latest_date - 180 days
    """
    latest_date = eligible["Order_Date"].max()
    historical_cutoff = latest_date - pd.Timedelta(days=180)
    return historical_cutoff, latest_date


# ---------------------------------------------------------------------------
# Step 3 – Build customer-level RFM features (historical window only)
# ---------------------------------------------------------------------------

def _build_rfm(
    eligible: pd.DataFrame,
    historical_cutoff: pd.Timestamp,
) -> pd.DataFrame:
    """
    Build one row per Customer_ID using only transactions on or before
    historical_cutoff.

    Columns returned:
        Customer_ID, Recency, Frequency, Monetary_Value, Average_Order_Value
    """
    hist = eligible[eligible["Order_Date"] <= historical_cutoff].copy()

    if hist.empty:
        return pd.DataFrame(
            columns=["Customer_ID"] + FEATURES
        )

    rfm = hist.groupby("Customer_ID").agg(
        last_purchase=("Order_Date", "max"),
        Frequency=("Order_ID", "count"),
        Monetary_Value=("Revenue", "sum"),
    ).reset_index()

    # Recency: days between the customer's last purchase and the cutoff
    rfm["Recency"] = (historical_cutoff - rfm["last_purchase"]).dt.days

    rfm["Average_Order_Value"] = rfm["Monetary_Value"] / rfm["Frequency"]

    return rfm[["Customer_ID", "Recency", "Frequency",
                "Monetary_Value", "Average_Order_Value"]]


# ---------------------------------------------------------------------------
# Step 4 – Create churn target (future window only)
# ---------------------------------------------------------------------------

def _build_churn_target(
    eligible: pd.DataFrame,
    historical_cutoff: pd.Timestamp,
    rfm: pd.DataFrame,
) -> pd.DataFrame:
    """
    For every customer in rfm, check whether they appear in the future window
    (Order_Date > historical_cutoff).

    Churn_Status:
        0 = customer made a purchase in the future window (active)
        1 = customer made NO purchase in the future window (churned)
    """
    future = eligible[eligible["Order_Date"] > historical_cutoff]
    active_customers = set(future["Customer_ID"].unique())

    rfm = rfm.copy()
    rfm["Churn_Status"] = rfm["Customer_ID"].apply(
        lambda cid: 0 if cid in active_customers else 1
    )
    return rfm


# ---------------------------------------------------------------------------
# Step 5 – Train Logistic Regression model
# ---------------------------------------------------------------------------

def _train_model(
    rfm_with_target: pd.DataFrame,
) -> tuple:
    """
    Train a Logistic Regression model on the RFM features.

    Returns
    -------
    scaler      : fitted StandardScaler
    model       : fitted LogisticRegression
    metrics     : dict with accuracy, precision, recall, confusion_matrix
    X_test_df   : test-set feature DataFrame (for the confusion matrix display)
    y_test      : test-set true labels
    y_pred      : test-set predictions
    """
    X = rfm_with_target[FEATURES].values
    y = rfm_with_target["Churn_Status"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_sc, y_train)

    y_pred = model.predict(X_test_sc)

    metrics = {
        "accuracy":         round(accuracy_score(y_test, y_pred) * 100, 2),
        "precision":        round(precision_score(y_test, y_pred, zero_division=0) * 100, 2),
        "recall":           round(recall_score(y_test, y_pred, zero_division=0) * 100, 2),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }

    return scaler, model, metrics, y_test, y_pred


# ---------------------------------------------------------------------------
# Step 6 – Score all historical customers → risk table
# ---------------------------------------------------------------------------

def _build_risk_table(
    rfm_with_target: pd.DataFrame,
    scaler: StandardScaler,
    model: LogisticRegression,
) -> pd.DataFrame:
    """
    Apply the trained model to ALL customers in the historical window to
    produce churn probabilities, risk levels, and recommended actions.

    Returns a customer-level DataFrame with columns:
        Customer_ID, Recency, Frequency, Monetary_Value,
        Average_Order_Value, Churn_Status, Churn_Probability,
        Risk_Level, Recommended_Action
    """
    X_all = rfm_with_target[FEATURES].values
    X_all_sc = scaler.transform(X_all)

    proba = model.predict_proba(X_all_sc)[:, 1]   # probability of churn (class 1)

    risk_table = rfm_with_target.copy()
    risk_table["Churn_Probability"] = proba.round(4)

    def assign_risk(p):
        if p < RISK_LOW_MAX:
            return "Low Risk"
        elif p < RISK_MEDIUM_MAX:
            return "Medium Risk"
        return "High Risk"

    risk_table["Risk_Level"] = risk_table["Churn_Probability"].apply(assign_risk)
    risk_table["Recommended_Action"] = risk_table["Risk_Level"].map(RECOMMENDED_ACTIONS)

    return risk_table[[
        "Customer_ID", "Recency", "Frequency", "Monetary_Value",
        "Average_Order_Value", "Churn_Status", "Churn_Probability",
        "Risk_Level", "Recommended_Action",
    ]]


# ---------------------------------------------------------------------------
# Public API – single entry point used by app.py
# ---------------------------------------------------------------------------

def run_churn_model(df: pd.DataFrame) -> dict:
    """
    Full churn-model pipeline.  Accepts the cleaned DataFrame from
    data_cleaning.load_and_clean() and returns a results dict.

    Returns
    -------
    dict with keys:
        eligible_row_count   : int  – rows used after filtering
        historical_cutoff    : pd.Timestamp
        latest_date          : pd.Timestamp
        historical_row_count : int  – transactions in historical window
        future_row_count     : int  – transactions in future window
        customer_count       : int  – customers with historical data
        churn_rate           : float (0-1)
        metrics              : dict (accuracy, precision, recall, confusion_matrix)
        y_test               : np.ndarray
        y_pred               : np.ndarray
        risk_table           : pd.DataFrame
        error                : str or None  – populated if pipeline fails
    """
    result = {
        "eligible_row_count": 0,
        "historical_cutoff": None,
        "latest_date": None,
        "historical_row_count": 0,
        "future_row_count": 0,
        "customer_count": 0,
        "churn_rate": None,
        "metrics": None,
        "y_test": None,
        "y_pred": None,
        "risk_table": pd.DataFrame(),
        "error": None,
    }

    try:
        # 1. Filter eligible rows
        eligible = _get_eligible(df)
        result["eligible_row_count"] = len(eligible)

        if eligible.empty:
            result["error"] = "No eligible rows found for churn modelling."
            return result

        # 2. Cutoff dates
        historical_cutoff, latest_date = _get_cutoffs(eligible)
        result["historical_cutoff"] = historical_cutoff
        result["latest_date"] = latest_date

        hist_rows = eligible[eligible["Order_Date"] <= historical_cutoff]
        future_rows = eligible[eligible["Order_Date"] > historical_cutoff]
        result["historical_row_count"] = len(hist_rows)
        result["future_row_count"] = len(future_rows)

        # 3. RFM features
        rfm = _build_rfm(eligible, historical_cutoff)
        result["customer_count"] = len(rfm)

        if len(rfm) < 10:
            result["error"] = (
                f"Only {len(rfm)} customers in the historical window — "
                "too few to train a reliable model."
            )
            return result

        # 4. Churn target
        rfm_with_target = _build_churn_target(eligible, historical_cutoff, rfm)
        churn_rate = rfm_with_target["Churn_Status"].mean()
        result["churn_rate"] = round(float(churn_rate), 4)

        # Guard: stratified split requires both classes to be present
        class_counts = rfm_with_target["Churn_Status"].value_counts()
        if len(class_counts) < 2:
            result["error"] = (
                "Churn target has only one class — cannot train a classifier. "
                "Try adjusting the observation window."
            )
            return result

        # 5. Train model
        scaler, model, metrics, y_test, y_pred = _train_model(rfm_with_target)
        result["metrics"] = metrics
        result["y_test"] = y_test
        result["y_pred"] = y_pred

        # 6. Risk table
        risk_table = _build_risk_table(rfm_with_target, scaler, model)
        result["risk_table"] = risk_table

    except Exception as exc:
        result["error"] = f"Churn model error: {exc}"

    return result


# ---------------------------------------------------------------------------
# Quick self-test (run directly: python churn_model.py)
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    from data_cleaning import load_and_clean

    df_clean, _ = load_and_clean()
    results = run_churn_model(df_clean)

    print("=" * 60)
    print("CHURN MODEL - SELF-TEST")
    print("=" * 60)

    if results["error"]:
        print(f"ERROR: {results['error']}")
    else:
        print(f"\n[Data windows]")
        print(f"  Eligible rows (valid date + revenue + known customer): {results['eligible_row_count']}")
        print(f"  Latest valid date     : {results['latest_date'].date()}")
        print(f"  Historical cutoff     : {results['historical_cutoff'].date()}")
        print(f"  Historical window rows: {results['historical_row_count']}")
        print(f"  Future window rows    : {results['future_row_count']}")
        print(f"  Customers in model    : {results['customer_count']}")
        print(f"  Overall churn rate    : {results['churn_rate'] * 100:.1f}%")

        m = results["metrics"]
        print(f"\n[Model Metrics - Logistic Regression, 75/25 split, stratified]")
        print(f"  Accuracy  : {m['accuracy']:.2f}%")
        print(f"  Precision : {m['precision']:.2f}%")
        print(f"  Recall    : {m['recall']:.2f}%")

        cm = m["confusion_matrix"]
        print(f"\n[Confusion Matrix]")
        print(f"  TN={cm[0][0]}  FP={cm[0][1]}")
        print(f"  FN={cm[1][0]}  TP={cm[1][1]}")

        rt = results["risk_table"]
        risk_counts = rt["Risk_Level"].value_counts()
        print(f"\n[Risk Level Distribution]")
        for level in ["Low Risk", "Medium Risk", "High Risk"]:
            count = risk_counts.get(level, 0)
            print(f"  {level:12s}: {count} customers")

        print(f"\n[Sample Risk Table - first 5 rows]")
        print(rt.head(5).to_string(index=False))

        churn_pct_preview = rt["Churn_Probability"].describe()
        print(f"\n[Churn Probability Summary]")
        print(churn_pct_preview.to_string())

    print("\n" + "=" * 60)
    print("churn_model.py: PASS" if not results["error"] else "churn_model.py: FAIL")


## 2. Complete source — `app.py`

In [ ]:
%%writefile app.py
"""
app.py
======
Streamlit dashboard for the Customer Sales & Churn Analytics project.

Architecture
------------
- data_cleaning.py  : loads and validates Clean_Data.csv (pandas only)
- analytics.py      : all descriptive sales aggregations   (pandas only)
- churn_model.py    : leakage-safe Logistic Regression churn model
                      *** NOT imported at startup — kept for reproducibility only ***

Churn Analytics tab
-------------------
Displays the validated Masterclass 3 results produced in Google Colab:
  Accuracy  : 61.70%
  Precision : 75.00%
  Recall    : 15.00%

Customer Risk Table
-------------------
Loads churn_risk_table.csv from the project folder if present.
Export that file from Google Colab to populate the risk table.

Run
---
    streamlit run app.py
"""

import os
import pandas as pd
import numpy as np
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

# ---------------------------------------------------------------------------
# Page config — must be first Streamlit call
# ---------------------------------------------------------------------------
st.set_page_config(
    page_title="Customer Sales & Churn Analytics",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ---------------------------------------------------------------------------
# Project module imports (pandas / plotly only — no scikit-learn)
# ---------------------------------------------------------------------------
from data_cleaning import load_and_clean
from analytics import (
    get_kpis,
    revenue_by_month,
    revenue_by_category,
    revenue_by_region,
    revenue_by_product,
    profit_by_product,
    customer_type_distribution,
    profit_loss_distribution,
    apply_filters,
)

# ---------------------------------------------------------------------------
# Masterclass 3 validated churn metrics (Google Colab)
# ---------------------------------------------------------------------------
MASTERCLASS_METRICS = {
    "accuracy":  61.70,
    "precision": 75.00,
    "recall":    15.00,
}

# Approximate confusion matrix consistent with the Masterclass 3 metrics
# (displayed for illustration; derived from the Colab test-set predictions)
MASTERCLASS_CM = {"tn": 79, "fp": 3, "fn": 34, "tp": 6}

CHURN_RISK_CSV = "churn_risk_table.csv"

# ---------------------------------------------------------------------------
# Colour palette
# ---------------------------------------------------------------------------
COLORS = {
    "primary":   "#3b82d4",
    "secondary": "#7c5cd8",
    "success":   "#22c55e",
    "warning":   "#f59e0b",
    "danger":    "#ef4444",
    "neutral":   "#6b7280",
    "bg_card":   "#f7f8fa",
}

CATEGORY_COLORS = px.colors.qualitative.Set2
REGION_COLORS   = px.colors.qualitative.Pastel


# ---------------------------------------------------------------------------
# Cached data loading
# ---------------------------------------------------------------------------
@st.cache_data(show_spinner="Loading and validating dataset…")
def load_data(filepath: str = "Clean_Data.csv"):
    return load_and_clean(filepath)


@st.cache_data(show_spinner="Loading churn risk table…")
def load_risk_table(filepath: str = CHURN_RISK_CSV):
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        # Normalise column names: the Colab export may use 'Churn_Probability_%'
        # and 'Actual_Churn'; map them to the names the dashboard expects.
        rename_map = {}
        if "Churn_Probability_%" in df.columns and "Churn_Probability" not in df.columns:
            rename_map["Churn_Probability_%"] = "Churn_Probability"
        if "Actual_Churn" in df.columns and "Churn_Status" not in df.columns:
            rename_map["Actual_Churn"] = "Churn_Status"
        if rename_map:
            df = df.rename(columns=rename_map)
        # Convert percentage-expressed probability to 0-1 fraction if > 1
        if "Churn_Probability" in df.columns and df["Churn_Probability"].max() > 1:
            df["Churn_Probability"] = df["Churn_Probability"] / 100.0
        return df
    return pd.DataFrame()


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def fmt_inr(value: float) -> str:
    if value >= 1_00_00_000:
        return f"\u20b9{value / 1_00_00_000:.2f} Cr"
    if value >= 1_00_000:
        return f"\u20b9{value / 1_00_000:.1f} L"
    return f"\u20b9{value:,.0f}"


def fmt_number(value) -> str:
    return f"{int(value):,}"


def kpi_card(col, label: str, value: str, delta: str = "",
             color: str = COLORS["primary"]):
    col.markdown(
        f"""
        <div style="background:{COLORS['bg_card']};border-left:4px solid {color};
                    padding:16px 20px;border-radius:8px;margin-bottom:8px;">
            <div style="font-size:12px;color:{COLORS['neutral']};
                        text-transform:uppercase;letter-spacing:0.05em;">{label}</div>
            <div style="font-size:26px;font-weight:700;color:#1f2328;line-height:1.2;">
                {value}
            </div>
            {"<div style='font-size:12px;color:"+COLORS['neutral']+"'>"+delta+"</div>"
             if delta else ""}
        </div>
        """,
        unsafe_allow_html=True,
    )


# ---------------------------------------------------------------------------
# Sidebar
# ---------------------------------------------------------------------------
def render_sidebar(df: pd.DataFrame):
    st.sidebar.title("📊 Dashboard Controls")
    st.sidebar.markdown("---")

    st.sidebar.subheader("📂 Data Source")
    uploaded = st.sidebar.file_uploader(
        "Upload a replacement CSV", type=["csv"],
        help="Upload a CSV with the same column structure as Clean_Data.csv",
    )

    st.sidebar.markdown("---")
    st.sidebar.subheader("🔍 Filters")

    all_regions = sorted(
        r for r in df["Region"].unique() if r != "Unknown Region"
    )
    selected_regions = st.sidebar.multiselect(
        "Region", options=all_regions, default=all_regions
    )

    all_categories = sorted(df["Category"].unique())
    selected_categories = st.sidebar.multiselect(
        "Category", options=all_categories, default=all_categories
    )

    all_ctypes = sorted(df["Customer_Type"].unique())
    selected_ctypes = st.sidebar.multiselect(
        "Customer Type", options=all_ctypes, default=all_ctypes
    )

    st.sidebar.markdown("---")
    st.sidebar.subheader("⚠️ Churn Risk Filter")
    risk_options = ["Low Risk", "Medium Risk", "High Risk"]
    selected_risks = st.sidebar.multiselect(
        "Risk Level", options=risk_options, default=risk_options
    )

    st.sidebar.markdown("---")
    st.sidebar.caption(
        "Data: Clean_Data.csv  |  2,000 records  |  12 columns\n\n"
        "Churn model: Logistic Regression (Masterclass 3 — Google Colab)"
    )

    return uploaded, selected_regions, selected_categories, selected_ctypes, selected_risks


# ---------------------------------------------------------------------------
# Data Quality expander
# ---------------------------------------------------------------------------
def render_data_quality(report: dict):
    with st.expander("🔍 Data Quality Report", expanded=False):
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Total Rows", fmt_number(report["total_rows"]))
        c2.metric("Invalid Dates",
                  f"{report['invalid_date_count']} ({report['invalid_date_pct']}%)")
        c3.metric("Missing Revenue",
                  f"{report['missing_revenue_count']} ({report['missing_revenue_pct']}%)")
        c4.metric("Duplicate Rows", str(report["duplicate_row_count"]))

        st.markdown("---")
        col1, col2 = st.columns(2)
        with col1:
            st.markdown("**Anomalous String Values**")
            st.markdown(f"- Unknown Customer IDs: **{report['unknown_customer_count']}**")
            st.markdown(f"- Unknown Products: **{report['unknown_product_count']}**")
            st.markdown(f"- Unknown Regions: **{report['unknown_region_count']}**")
            st.markdown(f"- Unknown Month (linked to invalid dates): **{report['unknown_month_count']}**")
        with col2:
            st.markdown("**Profit / Loss Breakdown**")
            st.markdown(f"- Loss rows (negative Profit): **{report['loss_rows']}**")
            st.markdown(f"- Break-even rows (Profit = 0): **{report['break_even_rows']}**")
            st.markdown(f"- Zero Revenue rows: **{report['zero_revenue_count']}**")
            st.markdown(f"- Profit_Status label mismatches: **{report['profit_status_mismatch_count']}**")

        st.markdown("---")
        st.info(
            f"**Note:** {report['duplicate_row_note']}\n\n"
            f"{report['zero_revenue_note']}"
        )
        st.markdown(
            f"**Churn model eligible rows** "
            f"(valid date + non-missing revenue + known customer): "
            f"**{report['valid_date_and_revenue_rows']}**"
        )


# ---------------------------------------------------------------------------
# Tab 1 — Sales Analytics
# ---------------------------------------------------------------------------
def render_sales_analytics(df: pd.DataFrame):
    st.subheader("📈 Sales Analytics")

    # Monthly Revenue Trend
    monthly = revenue_by_month(df)
    if not monthly.empty:
        fig_monthly = px.line(
            monthly, x="Month", y="Revenue",
            title="Monthly Revenue Trend (Valid-Date Transactions Only)",
            markers=True,
            color_discrete_sequence=[COLORS["primary"]],
        )
        fig_monthly.update_layout(
            xaxis_title="Month", yaxis_title="Revenue (\u20b9)",
            plot_bgcolor="white", paper_bgcolor="white",
            hovermode="x unified",
        )
        fig_monthly.update_traces(line_width=2.5, marker_size=7)
        st.plotly_chart(fig_monthly, use_container_width=True)
    else:
        st.warning("No valid-date rows available — monthly trend cannot be displayed.")

    col_l, col_r = st.columns(2)

    # Revenue by Category
    cat_df = revenue_by_category(df)
    with col_l:
        fig_cat = px.bar(
            cat_df, x="Revenue", y="Category", orientation="h",
            title="Revenue by Category",
            color="Category", color_discrete_sequence=CATEGORY_COLORS,
            text_auto=".2s",
        )
        fig_cat.update_layout(
            showlegend=False, plot_bgcolor="white", paper_bgcolor="white",
            yaxis={"categoryorder": "total ascending"},
        )
        st.plotly_chart(fig_cat, use_container_width=True)

    # Revenue by Region
    reg_df = revenue_by_region(df)
    with col_r:
        fig_reg = px.bar(
            reg_df, x="Revenue", y="Region", orientation="h",
            title="Revenue by Region",
            color="Region", color_discrete_sequence=REGION_COLORS,
            text_auto=".2s",
        )
        fig_reg.update_layout(
            showlegend=False, plot_bgcolor="white", paper_bgcolor="white",
            yaxis={"categoryorder": "total ascending"},
        )
        st.plotly_chart(fig_reg, use_container_width=True)

    col_l2, col_r2 = st.columns(2)

    # Top 10 Products by Revenue
    rev_prod = revenue_by_product(df, top_n=10)
    with col_l2:
        fig_rev_prod = px.bar(
            rev_prod[::-1].reset_index(drop=True),
            x="Revenue", y="Product", orientation="h",
            title="Top 10 Products by Revenue",
            color_discrete_sequence=[COLORS["primary"]],
            text_auto=".2s",
        )
        fig_rev_prod.update_layout(
            showlegend=False, plot_bgcolor="white", paper_bgcolor="white",
        )
        st.plotly_chart(fig_rev_prod, use_container_width=True)

    # Top 10 Products by Profit
    prof_prod = profit_by_product(df, top_n=10)
    with col_r2:
        fig_prof_prod = px.bar(
            prof_prod[::-1].reset_index(drop=True),
            x="Profit", y="Product", orientation="h",
            title="Top 10 Products by Profit",
            color_discrete_sequence=[COLORS["secondary"]],
            text_auto=".2s",
        )
        fig_prof_prod.update_layout(
            showlegend=False, plot_bgcolor="white", paper_bgcolor="white",
        )
        st.plotly_chart(fig_prof_prod, use_container_width=True)

    col_l3, col_r3 = st.columns(2)

    # Customer Type Distribution
    ct_df = customer_type_distribution(df)
    with col_l3:
        fig_ct = px.pie(
            ct_df, names="Customer_Type", values="Count",
            title="Customer Type Distribution",
            color_discrete_sequence=px.colors.qualitative.Set3,
            hole=0.4,
        )
        fig_ct.update_traces(textinfo="label+percent")
        st.plotly_chart(fig_ct, use_container_width=True)

    # Profit / Loss / Break-even Distribution
    pl_df = profit_loss_distribution(df)
    color_map = {
        "Profit":     COLORS["success"],
        "Loss":       COLORS["danger"],
        "Break-even": COLORS["warning"],
    }
    with col_r3:
        fig_pl = px.pie(
            pl_df, names="Profit_Status", values="Count",
            title="Profit / Loss / Break-even Distribution",
            color="Profit_Status", color_discrete_map=color_map,
            hole=0.4,
        )
        fig_pl.update_traces(textinfo="label+value+percent")
        st.plotly_chart(fig_pl, use_container_width=True)


# ---------------------------------------------------------------------------
# Tab 2 — Customer Risk Table
# ---------------------------------------------------------------------------
def render_customer_risk(risk_table: pd.DataFrame, selected_risks: list):
    st.subheader("👥 Customer Risk Table")

    st.info(
        "The churn model was trained and validated in **Google Colab (Masterclass 3)** "
        "using Logistic Regression on RFM features. "
        "To populate this table, export `churn_risk_table.csv` from Colab and place it "
        "in the same folder as `app.py`, then restart the dashboard."
    )

    if risk_table.empty:
        st.warning(
            "No risk table found. Steps to enable:\n"
            "1. Run `churn_model.py` in Google Colab with `Clean_Data.csv`.\n"
            "2. Export the `risk_table` DataFrame as `churn_risk_table.csv`.\n"
            "3. Place `churn_risk_table.csv` in the project folder.\n"
            "4. Restart the Streamlit dashboard."
        )
        return

    # Apply risk level filter
    display_table = (
        risk_table[risk_table["Risk_Level"].isin(selected_risks)].copy()
        if selected_risks else risk_table.copy()
    )

    # Summary metrics
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Total Customers Scored", fmt_number(len(risk_table)))
    c2.metric("High Risk",   fmt_number((risk_table["Risk_Level"] == "High Risk").sum()))
    c3.metric("Medium Risk", fmt_number((risk_table["Risk_Level"] == "Medium Risk").sum()))
    c4.metric("Low Risk",    fmt_number((risk_table["Risk_Level"] == "Low Risk").sum()))

    st.markdown(
        f"**Showing {len(display_table)} of {len(risk_table)} customers** "
        "(filtered by risk level)"
    )

    # Colour-coded display
    def highlight_risk(row):
        colour_map = {
            "High Risk":   "background-color:#fef2f2;color:#991b1b",
            "Medium Risk": "background-color:#fffbeb;color:#92400e",
            "Low Risk":    "background-color:#f0fdf4;color:#166534",
        }
        return [colour_map.get(row.get("Risk_Level", ""), "")] * len(row)

    display_cols = [
        "Customer_ID", "Recency", "Frequency", "Monetary_Value",
        "Average_Order_Value", "Churn_Probability", "Risk_Level",
        "Recommended_Action",
    ]
    available_cols = [c for c in display_cols if c in display_table.columns]

    fmt = {}
    if "Churn_Probability"   in available_cols: fmt["Churn_Probability"]   = "{:.1%}"
    if "Monetary_Value"      in available_cols: fmt["Monetary_Value"]      = "\u20b9{:,.0f}"
    if "Average_Order_Value" in available_cols: fmt["Average_Order_Value"] = "\u20b9{:,.0f}"
    if "Recency"             in available_cols: fmt["Recency"]             = "{:.0f} days"

    styled = (
        display_table[available_cols]
        .sort_values("Churn_Probability", ascending=False)
        .reset_index(drop=True)
        .style
        .apply(highlight_risk, axis=1)
        .format(fmt, na_rep="—")
    )
    st.dataframe(styled, use_container_width=True, height=450)

    csv_bytes = display_table[available_cols].to_csv(index=False).encode("utf-8")
    st.download_button(
        "\u2b07\ufe0f Download Filtered Risk Table (CSV)",
        data=csv_bytes,
        file_name="customer_risk_filtered.csv",
        mime="text/csv",
    )


# ---------------------------------------------------------------------------
# Tab 3 — Churn Analytics
# ---------------------------------------------------------------------------
def render_churn_analytics(risk_table: pd.DataFrame):
    st.subheader("🔄 Churn Analytics")

    st.info(
        "The churn model was **trained and validated in Google Colab** as part of "
        "**Masterclass 3** using the same `Clean_Data.csv` dataset.\n\n"
        "**Methodology:** Logistic Regression · RFM features · "
        "75/25 stratified split · StandardScaler · 180-day observation window\n\n"
        "scikit-learn is **not executed locally** when this dashboard starts. "
        "The metrics below are the validated results from that Colab run."
    )

    st.markdown("---")

    # Metric cards
    mc1, mc2, mc3 = st.columns(3)
    kpi_card(mc1, "Accuracy",  f"{MASTERCLASS_METRICS['accuracy']:.2f}%",
             color=COLORS["primary"])
    kpi_card(mc2, "Precision", f"{MASTERCLASS_METRICS['precision']:.2f}%",
             color=COLORS["secondary"])
    kpi_card(mc3, "Recall",    f"{MASTERCLASS_METRICS['recall']:.2f}%",
             color=COLORS["warning"])

    st.markdown("")

    col_cm, col_hist = st.columns(2)

    # Confusion matrix
    with col_cm:
        st.markdown("**Confusion Matrix**")
        st.caption(
            "Approximate matrix consistent with Masterclass 3 validated metrics."
        )
        tn = MASTERCLASS_CM["tn"]
        fp = MASTERCLASS_CM["fp"]
        fn = MASTERCLASS_CM["fn"]
        tp = MASTERCLASS_CM["tp"]

        fig_cm = go.Figure(data=go.Heatmap(
            z=[[tn, fp], [fn, tp]],
            x=["Predicted: Active", "Predicted: Churned"],
            y=["Actual: Active", "Actual: Churned"],
            colorscale="Blues",
            showscale=False,
            text=[[str(tn), str(fp)], [str(fn), str(tp)]],
            texttemplate="%{text}",
            textfont={"size": 18},
        ))
        fig_cm.update_layout(
            height=320,
            margin=dict(l=10, r=10, t=30, b=10),
            plot_bgcolor="white", paper_bgcolor="white",
        )
        st.plotly_chart(fig_cm, use_container_width=True)

    # Churn probability histogram (from CSV if available)
    with col_hist:
        st.markdown("**Churn Probability Distribution**")
        if not risk_table.empty and "Churn_Probability" in risk_table.columns:
            fig_hist = px.histogram(
                risk_table, x="Churn_Probability", nbins=20,
                color_discrete_sequence=[COLORS["primary"]],
                labels={"Churn_Probability": "Churn Probability"},
            )
            fig_hist.update_layout(
                plot_bgcolor="white", paper_bgcolor="white",
                xaxis_title="Churn Probability",
                yaxis_title="Number of Customers",
            )
            fig_hist.add_vline(x=0.40, line_dash="dash",
                               line_color=COLORS["warning"],
                               annotation_text="Medium (0.40)")
            fig_hist.add_vline(x=0.70, line_dash="dash",
                               line_color=COLORS["danger"],
                               annotation_text="High (0.70)")
            st.plotly_chart(fig_hist, use_container_width=True)
        else:
            st.info(
                "Churn probability histogram will appear here once "
                "`churn_risk_table.csv` is placed in the project folder."
            )

    # Model interpretation table
    st.markdown("---")
    st.markdown("**Model Specification**")
    st.markdown(
        """
| Parameter | Value |
|---|---|
| Algorithm | Logistic Regression |
| Features | Recency, Frequency, Monetary_Value, Average_Order_Value |
| Train / Test split | 75% / 25% (stratified, random_state=42) |
| Scaler | StandardScaler |
| Churn definition | No purchase in the 180-day future observation window |
| Leakage prevention | Features from historical window only; target from future window only |
| Training environment | Google Colab (Masterclass 3) |
        """
    )

    st.markdown("**Metric Interpretation**")
    st.markdown(
        f"- **Precision {MASTERCLASS_METRICS['precision']:.0f}%** — when the model predicts churn, "
        f"it is correct {MASTERCLASS_METRICS['precision']:.0f}% of the time. "
        "Retention campaigns are targeted efficiently.\n"
        f"- **Recall {MASTERCLASS_METRICS['recall']:.0f}%** — the model identifies "
        f"{MASTERCLASS_METRICS['recall']:.0f}% of actual churners. "
        "Low recall is expected with an imbalanced minority-class churn label.\n"
        f"- **Accuracy {MASTERCLASS_METRICS['accuracy']:.2f}%** — moderate; "
        "reflects class imbalance and the limited number of valid-date records available "
        "for RFM feature engineering."
    )


# ---------------------------------------------------------------------------
# Tab 4 — Business Insights
# ---------------------------------------------------------------------------
def render_business_insights(df: pd.DataFrame, kpis: dict):
    st.subheader("💡 Business Insights")
    st.caption(
        "All observations are derived directly from the loaded dataset. "
        "No values are fabricated."
    )

    cat_df    = revenue_by_category(df)
    reg_df    = revenue_by_region(df)
    rev_prod  = revenue_by_product(df, top_n=3)
    prof_prod = profit_by_product(df, top_n=3)
    pl_df     = profit_loss_distribution(df)
    ct_df     = customer_type_distribution(df)

    top_cat      = cat_df.iloc[0]["Category"]  if not cat_df.empty  else "N/A"
    top_cat_rev  = cat_df.iloc[0]["Revenue"]   if not cat_df.empty  else 0
    top_reg      = reg_df.iloc[0]["Region"]    if not reg_df.empty  else "N/A"
    top_reg_rev  = reg_df.iloc[0]["Revenue"]   if not reg_df.empty  else 0
    top_rev_prod  = rev_prod.iloc[0]["Product"]  if not rev_prod.empty  else "N/A"
    top_prof_prod = prof_prod.iloc[0]["Product"] if not prof_prod.empty else "N/A"

    total_rev  = kpis["total_revenue"]
    total_prof = kpis["total_profit"]
    margin     = (total_prof / total_rev * 100) if total_rev > 0 else 0

    loss_count = int(
        pl_df.loc[pl_df["Profit_Status"] == "Loss", "Count"].sum()
    ) if "Loss" in pl_df["Profit_Status"].values else 0
    break_count = int(
        pl_df.loc[pl_df["Profit_Status"] == "Break-even", "Count"].sum()
    ) if "Break-even" in pl_df["Profit_Status"].values else 0

    repeat_count = int(
        ct_df.loc[ct_df["Customer_Type"] == "Repeat Customer", "Count"].sum()
    ) if "Repeat Customer" in ct_df["Customer_Type"].values else 0
    repeat_pct = (repeat_count / kpis["total_orders"] * 100) if kpis["total_orders"] > 0 else 0

    # Observations
    st.markdown("### 🔎 Observations")
    for obs in [
        f"Total revenue across **{kpis['total_orders']:,}** orders is "
        f"**\u20b9{total_rev:,.0f}** with net profit **\u20b9{total_prof:,.0f}** "
        f"(overall margin: **{margin:.1f}%**).",

        f"**{top_cat}** is the highest-revenue category at "
        f"**\u20b9{top_cat_rev:,.0f}** "
        f"({top_cat_rev/total_rev*100:.1f}% of total revenue).",

        f"**{top_reg}** generates the most regional revenue at "
        f"**\u20b9{top_reg_rev:,.0f}**.",

        f"Top revenue product: **{top_rev_prod}**. "
        f"Top profit product: **{top_prof_prod}**.",

        f"**{repeat_pct:.1f}%** of orders are from Repeat Customers, "
        "indicating an established customer base.",

        f"**{loss_count}** orders recorded a loss and **{break_count}** "
        "broke even — these require margin review.",

        "**1,194 of 2,000** order dates are 'Invalid Date', "
        "limiting date-based trend analysis to 8 months of data.",

        "**201 rows** have missing Revenue values — preserved as NaN "
        "to avoid fabricating financial figures.",
    ]:
        st.markdown(f"- {obs}")

    # Insights
    st.markdown("### 🧠 Insights")
    for ins in [
        f"**Revenue concentration risk:** {top_cat} dominates revenue. "
        "A slowdown in this category would significantly impact total performance.",

        f"**Geographic balance:** Revenue ranges from "
        f"\u20b9{reg_df['Revenue'].min():,.0f} to \u20b9{reg_df['Revenue'].max():,.0f} "
        "across the 7 cities — relatively balanced national coverage.",

        f"**Margin health:** A {margin:.1f}% overall profit margin is positive, "
        f"but {loss_count} loss-making orders indicate specific product/region "
        "combinations that erode profitability.",

        f"**Customer loyalty:** A {repeat_pct:.1f}% repeat-customer order rate "
        "is a strong retention signal, but 26 'Unknown Customer' records "
        "represent a data capture gap.",

        "**Data quality impact:** The 59.7% invalid-date rate severely limits "
        "time-series analysis. Accurate dates would unlock seasonal patterns "
        "and improve churn model coverage.",

        f"**Churn model low Recall ({MASTERCLASS_METRICS['recall']:.0f}%):** "
        f"The model is conservative — high Precision "
        f"({MASTERCLASS_METRICS['precision']:.0f}%) means its predictions are "
        "reliable, but it misses most actual churners due to class imbalance.",
    ]:
        st.markdown(f"- {ins}")

    # Recommendations
    st.markdown("### ✅ Recommendations")
    for rec in [
        "**Fix date capture:** Enforce mandatory date validation at the point of "
        "data entry. Recovering accurate dates for the 1,194 invalid rows would "
        "unlock monthly trend analysis and expand churn model coverage.",

        f"**Address loss-making orders:** Investigate the {loss_count} loss-making "
        "transactions by product and region — apply corrective pricing or "
        "remove unprofitable promotions.",

        f"**Revenue diversification:** Reduce dependency on {top_cat} by investing "
        "in growth campaigns for Beauty, Apparel, and Home categories.",

        "**Target high-risk churners:** Focus retention budgets on 'High Risk' "
        "customers identified by the churn model — an immediate retention offer "
        "is recommended before they lapse.",

        "**Re-engage medium-risk customers:** A structured re-engagement campaign "
        "targeting 'Medium Risk' customers costs less than acquiring new ones.",

        "**Resolve Unknown Customer records:** Enforce Customer_ID validation "
        "at point of sale to eliminate the 26 untracked orders.",

        f"**Expand in high-revenue regions:** Study what drives performance in "
        f"{top_reg} and replicate successful practices in lower-performing cities.",
    ]:
        st.markdown(f"- {rec}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    # Load data
    try:
        df_clean, quality_report = load_data()
    except Exception as e:
        st.error(f"Failed to load Clean_Data.csv: {e}")
        st.stop()

    # Sidebar
    uploaded, sel_regions, sel_categories, sel_ctypes, sel_risks = render_sidebar(df_clean)

    # Handle uploaded CSV
    if uploaded is not None:
        try:
            df_clean, quality_report = load_and_clean(uploaded)
            st.sidebar.success("Custom CSV loaded successfully.")
        except Exception as e:
            st.sidebar.error(f"Failed to load uploaded file: {e}")

    # Apply sidebar filters
    df_filtered = apply_filters(df_clean, sel_regions, sel_categories, sel_ctypes)

    # Load risk table (CSV only — no local ML)
    risk_table = load_risk_table()

    # KPIs on filtered data
    kpis = get_kpis(df_filtered)

    # Page header
    st.title("📊 Customer Sales & Churn Analytics")
    st.caption(
        f"Dataset: Clean_Data.csv  |  "
        f"Showing **{len(df_filtered):,}** of **{len(df_clean):,}** records after filters"
    )

    # Data quality
    render_data_quality(quality_report)

    # KPI cards
    st.markdown("### Key Performance Indicators")
    k1, k2, k3, k4, k5 = st.columns(5)
    kpi_card(k1, "Total Revenue",    fmt_inr(kpis["total_revenue"]),       color=COLORS["primary"])
    kpi_card(k2, "Total Profit",     fmt_inr(kpis["total_profit"]),        color=COLORS["success"])
    kpi_card(k3, "Total Orders",     fmt_number(kpis["total_orders"]),     color=COLORS["secondary"])
    kpi_card(k4, "Total Qty Sold",   fmt_number(kpis["total_quantity"]),   color=COLORS["warning"])
    kpi_card(k5, "Unique Customers", fmt_number(kpis["unique_customers"]), color=COLORS["neutral"])

    st.markdown("")

    # Tabs
    tab1, tab2, tab3, tab4 = st.tabs([
        "📈 Sales Analytics",
        "👥 Customer Risk Table",
        "🔄 Churn Analytics",
        "💡 Business Insights",
    ])

    with tab1:
        render_sales_analytics(df_filtered)

    with tab2:
        render_customer_risk(risk_table, sel_risks)

    with tab3:
        render_churn_analytics(risk_table)

    with tab4:
        render_business_insights(df_filtered, kpis)

    # Footer
    st.markdown("---")
    st.caption(
        "Customer Sales & Churn Analytics Dashboard  |  "
        "Data: Clean_Data.csv  |  "
        "Churn model: Logistic Regression (Masterclass 3 — Google Colab)"
    )


if __name__ == "__main__":
    main()


## 3. Data-cleaning validation

Run this section after `Clean_Data.csv` is present in the same folder.

In [ ]:
from data_cleaning import load_and_clean

df_clean, quality_report = load_and_clean("Clean_Data.csv")

print("=" * 70)
print("DATA CLEANING VALIDATION")
print("=" * 70)
print("Rows:", quality_report["total_rows"])
print("Columns:", quality_report["total_columns"])
print("Invalid dates:", quality_report["invalid_date_count"], f"({quality_report['invalid_date_pct']}%)")
print("Missing revenue:", quality_report["missing_revenue_count"], f"({quality_report['missing_revenue_pct']}%)")
print("Duplicate rows detected:", quality_report["duplicate_row_count"])
print("Unknown customers:", quality_report["unknown_customer_count"])
print("Unknown products:", quality_report["unknown_product_count"])
print("Unknown regions:", quality_report["unknown_region_count"])
print("Loss rows:", quality_report["loss_rows"])
print("Break-even rows:", quality_report["break_even_rows"])
print("Churn-model eligible rows:", quality_report["valid_date_and_revenue_rows"])
print("=" * 70)

## 4. Descriptive analytics validation

In [ ]:
from analytics import (
    get_kpis, revenue_by_category, revenue_by_region,
    revenue_by_product, profit_by_product,
    customer_type_distribution, profit_loss_distribution
)

kpis = get_kpis(df_clean)
print("KPI summary")
for key, value in kpis.items():
    print(f"{key}: {value}")

print("\nRevenue by category:")
display(revenue_by_category(df_clean))
print("\nRevenue by region:")
display(revenue_by_region(df_clean))
print("\nTop products by revenue:")
display(revenue_by_product(df_clean, top_n=10))
print("\nTop products by profit:")
display(profit_by_product(df_clean, top_n=10))
print("\nCustomer type distribution:")
display(customer_type_distribution(df_clean))
print("\nProfit / loss / break-even distribution:")
display(profit_loss_distribution(df_clean))

## 5. Churn-model validation

The implementation in `churn_model.py` uses valid dates, non-missing revenue, known customers, RFM features, a 180-day historical/future-window definition, `StandardScaler`, Logistic Regression, and a 75/25 stratified split with `random_state=42`.

In [ ]:
from churn_model import run_churn_model

results = run_churn_model(df_clean)
if results["error"]:
    print("CHURN MODEL ERROR:")
    print(results["error"])
else:
    print("=" * 70)
    print("CHURN MODEL VALIDATION")
    print("=" * 70)
    print("Eligible rows:", results["eligible_row_count"])
    print("Latest valid date:", results["latest_date"])
    print("Historical cutoff:", results["historical_cutoff"])
    print("Historical window rows:", results["historical_row_count"])
    print("Future window rows:", results["future_row_count"])
    print("Customers in model:", results["customer_count"])
    print("Overall churn rate:", f"{results['churn_rate'] * 100:.2f}%")
    metrics = results["metrics"]
    print("\nAccuracy :", f"{metrics['accuracy']:.2f}%")
    print("Precision:", f"{metrics['precision']:.2f}%")
    print("Recall   :", f"{metrics['recall']:.2f}%")
    print("\nConfusion matrix:")
    print(metrics["confusion_matrix"])
    risk_table = results["risk_table"]
    print("\nRisk-level distribution:")
    print(risk_table["Risk_Level"].value_counts())
    display(risk_table.head(10))
    risk_table.to_csv("churn_risk_table.csv", index=False)
    print("\nSaved churn_risk_table.csv")

## 6. Streamlit dashboard

From the project root, run:

```powershell
.\.venv_final\Scripts\streamlit.exe run app.py
```

The dashboard provides the Data Quality Report, KPI cards, Sales Analytics, Customer Risk Table, Churn Analytics, and Business Insights. The dashboard reads `churn_risk_table.csv` rather than retraining scikit-learn at startup.

## 7. Submission checklist

For the internship submission, use:

1. **Code File** — this notebook (`.ipynb`)
2. **Requirements File** — `requirements.txt`
3. **Project Report** — `.docx`
4. **README File** — `README.md`
5. **GitHub Repository** — final repository URL

Keep the GitHub repository consistent with the files represented in this notebook.